# 02. Анализ результатов A/B-теста

Ожидаемый входной файл: `data/final_results_to_analyze.xlsx`. Он не включён в исходные материалы, поэтому ноутбук сначала проверяет его наличие и не создаёт искусственные результаты.

В Excel ожидаются поля `date`, `user_id`, `ab_group`, `is_view_banner`, `cnt_view_banner`, `is_click_banner`, `cnt_click_banner`, `total_watch_time`, `is_add_fav`, `cnt_add_fav`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
RESULTS_PATH = ROOT / "data" / "final_results_to_analyze.xlsx"
if not RESULTS_PATH.exists():
    print(f"Файл не найден: {RESULTS_PATH}")
    print("Поместите final_results_to_analyze.xlsx в папку data и запустите ноутбук повторно.")
    results = None
else:
    results = pd.read_excel(RESULTS_PATH)
    display(results.head())

In [ ]:
if results is not None:
    required = {"user_id", "ab_group", "cnt_view_banner", "cnt_click_banner",
                "total_watch_time", "cnt_add_fav"}
    missing = required - set(results.columns)
    if missing:
        raise ValueError(f"В файле отсутствуют колонки: {sorted(missing)}")

    # Если в файле несколько строк на пользователя, агрегируем до пользователя.
    user = results.groupby(["user_id", "ab_group"], as_index=False).agg(
        views=("cnt_view_banner", "sum"),
        clicks=("cnt_click_banner", "sum"),
        watch_time=("total_watch_time", "sum"),
        adds_to_fav=("cnt_add_fav", "sum")
    )
    user["ctr"] = np.where(user["views"] > 0, user["clicks"] / user["views"], np.nan)
    user["avg_watch_time"] = np.where(user["clicks"] > 0, user["watch_time"] / user["clicks"], np.nan)
    user["cr_fav"] = np.where(user["clicks"] > 0, user["adds_to_fav"] / user["clicks"], np.nan)
    user["arpu"] = user["watch_time"]  # watch time на пользователя за весь период
    display(user.head())

In [ ]:
if results is not None:
    metrics = ["ctr", "avg_watch_time", "cr_fav", "arpu"]
    summary = user.groupby("ab_group")[metrics].agg(["mean", "var", "count"])
    display(summary)

In [ ]:
def bootstrap_diff(a, b, n_boot=5000, seed=42):
    """Bootstrap CI для разницы средних: test - control."""
    a = np.asarray(pd.Series(a).dropna(), dtype=float)
    b = np.asarray(pd.Series(b).dropna(), dtype=float)
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        diffs[i] = rng.choice(b, len(b), replace=True).mean() - rng.choice(a, len(a), replace=True).mean()
    return float(b.mean() - a.mean()), float(np.quantile(diffs, .025)), float(np.quantile(diffs, .975))

if results is not None:
    # Уточните названия групп в соответствии с фактическим Excel.
    print("Группы в файле:", user["ab_group"].dropna().unique())
    groups = list(user["ab_group"].dropna().unique())
    if len(groups) != 2:
        print("Для сравнения нужны ровно две группы.")
    else:
        control_name, test_name = groups[0], groups[1]
        rows = []
        for metric in metrics:
            control = user.loc[user["ab_group"].eq(control_name), metric].dropna()
            test = user.loc[user["ab_group"].eq(test_name), metric].dropna()
            if len(control) >= 2 and len(test) >= 2:
                t_stat, p_value = stats.ttest_ind(test, control, equal_var=False, nan_policy="omit")
                diff, ci_low, ci_high = bootstrap_diff(control, test)
                relative_lift = (test.mean()/control.mean()-1)*100 if control.mean() != 0 else np.nan
            else:
                t_stat, p_value, diff, ci_low, ci_high, relative_lift = [np.nan]*6
            rows.append({"metric":metric, "control_mean":control.mean(), "test_mean":test.mean(),
                         "absolute_diff_test_minus_control":diff, "relative_lift_pct":relative_lift,
                         "bootstrap_ci_2.5%":ci_low, "bootstrap_ci_97.5%":ci_high,
                         "welch_t_test_p_value":p_value})
        test_results = pd.DataFrame(rows)
        display(test_results)
        test_results.to_csv(ROOT / "reports" / "ab_test_metrics_and_tests.csv", index=False)

In [ ]:
if results is not None and len(user["ab_group"].dropna().unique()) == 2:
    for metric in metrics:
        plt.figure(figsize=(6,4))
        user.boxplot(column=metric, by="ab_group", grid=False)
        plt.title(f"Сравнение групп: {metric}")
        plt.suptitle("")
        plt.xlabel("A/B-группа")
        plt.ylabel(metric)
        plt.tight_layout()
        plt.show()

## Интерпретация

Ключевая метрика — `avg_watch_time`. Положительный результат фиксируется, если среднее значение в тестовой группе выше контрольного и статистический тест показывает `p < 0.05`; дополнительно полезно проверить, что bootstrap-доверительный интервал разницы не включает ноль. Вторичные метрики и защитные показатели рассматриваются отдельно.

Перед финальным выводом убедитесь, что:
1. значения `ab_group` однозначно сопоставлены контролю и тесту;
2. `total_watch_time` измеряется в согласованных единицах;
3. знаменатели и правила включения пользователей определены до просмотра результата;
4. если метрики сильно скошены, bootstrap предпочтительнее обычного t-теста, а множественные проверки вторичных метрик интерпретируются осторожно.